do pliku `build.gradle(Module)`

```kotlin
dependencies {

    implementation("androidx.lifecycle:lifecycle-viewmodel-compose:2.9.2")

    ...
}
```

```kotlin
class MainActivity : ComponentActivity() {
    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        enableEdgeToEdge()
        setContent {
            StateInShatrdInBasicExampleTheme {
                StreamerBroadcastScreen()
            }
        }
    }
}

// =================================================================================
// --- ViewModel Streamera ---
// =================================================================================

class StreamerViewModel : ViewModel() {

    // 1. ZIMNY STRUMIEŃ: Gotowy plik wideo "Top 5 Klipów"
    private val topPlaysVideo: Flow<String> = flow {
        val clips = listOf("Klip #1: Stream 3!", "Klip #2: Stream 1!",
            "Klip #3: Stream 5!", "Klip #4: Stream 12!")
        for (clip in clips) {
            emit(clip)
            delay(3000)
        }
    }

    // 2. GORĄCY STATEFLOW: "Kanał Twitch" retransmitujący wideo na żywo
    val liveTopPlays: StateFlow<String> = topPlaysVideo
        .stateIn(
            scope = viewModelScope,
            started = SharingStarted.WhileSubscribed(5000),
            initialValue = "Retransmisja rozpocznie się za chwilę..."
        )


    // 1. ZIMNY STRUMIEŃ: Gotowa ścieżka dźwiękowa z alertami
    private val alertAudioTrack: Flow<String> = flow {
        delay(3000)
        emit("NOWY SUB!")
        delay(7000)
        emit("NOWA DONACJA!")
    }

    // 2. GORĄCY SHAREDFLOW: System powiadomień "na żywo"
    val liveAlerts: SharedFlow<String> = alertAudioTrack
        .shareIn(
            scope = viewModelScope,
            started = SharingStarted.WhileSubscribed(5000)
        )
}

// =================================================================================
// --- UI (Widok) ---
// =================================================================================

@OptIn(ExperimentalMaterial3Api::class)
@Composable
fun StreamerBroadcastScreen(viewModel: StreamerViewModel = viewModel()) {
    // Widz subskrybuje STAN (co jest aktualnie na ekranie)
    val currentPlay by viewModel.liveTopPlays.collectAsStateWithLifecycle()

    val snackbarHostState = remember { SnackbarHostState() }

    // Widz subskrybuje ZDARZENIA (powiadomienia)
    LaunchedEffect(Unit) {
        viewModel.liveAlerts.collect { alert ->
            snackbarHostState.showSnackbar(alert)
        }
    }

    Scaffold(snackbarHost = { SnackbarHost(snackbarHostState) }) { padding ->
        Column(
            modifier = Modifier.fillMaxSize().padding(padding).padding(16.dp),
            horizontalAlignment = Alignment.CenterHorizontally,
            verticalArrangement = Arrangement.Center
        ) {
            Text("Retransmisja 'Top 5 Klipów'", style = MaterialTheme.typography.headlineMedium)
            Text("(przykład stateIn)", style = MaterialTheme.typography.labelMedium)

            Card(modifier = Modifier.padding(vertical = 16.dp)) {
                Text(
                    text = currentPlay,
                    style = MaterialTheme.typography.titleLarge,
                    textAlign = TextAlign.Center,
                    modifier = Modifier.padding(24.dp)
                )
            }

            Text(
                "Powiadomienia z transmisji (przykład shareIn) pojawią się na dole ekranu.",
                style = MaterialTheme.typography.bodyMedium,
                textAlign = TextAlign.Center
            )
        }
    }
}
```